# NB_LIBRO_09 · Figuras y experimentos del capítulo 7

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 7, «Caso palma africana: clasificar cinco estados del racimo»

Este notebook:

1. evalúa los nueve clasificadores YOLO11 del proyecto (tamaños n, s y m, semillas 2026 a 2028) sobre validación y prueba, y guarda la probabilidad de cada clase para cada imagen;
2. muestra los errores del primer modelo y las imágenes de prueba en las que más modelos fallan;
3. muestra qué parte de la fotografía recibe realmente el modelo;
4. clasifica una imagen de cada clase y mide el tiempo por imagen en GPU y en CPU;
5. prueba el modelo con imágenes que no son racimos;
6. recoge las curvas de entrenamiento guardadas por el proyecto;
7. hace un experimento nuevo: entrena YOLO11m con las clases escasas sobremuestreadas (tres semillas).

**Usa (solo lectura):** `PALMA_AFRICANA_YOLO` (datos procesados, `05_MODELOS` y resultados) y una imagen de `AgroUTMACH_PIX`. No modifica nada de los proyectos.

**Produce:** cinco figuras, `valores_cap07.json` y varios CSV en `AGROVISION_LIBRO`. Al final descarga un zip.

**Cómo ejecutarlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (L4 o T4) y *Ejecutar todo*. Tarda de 12 a 18 minutos. Si Colab se desconecta, vuelva a ejecutar todo: los entrenamientos terminados se reutilizan. Suba al chat el zip que se descarga al final.

In [ ]:
%pip install -q ultralytics==8.4.172 scikit-learn

In [ ]:
import os, re, json, math, zipfile, time, shutil, gc, random, traceback, platform
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content'))
PALMA = BASE / 'PALMA_AFRICANA_YOLO'
DATOS = PALMA / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
DIR_MOD = PALMA / '05_MODELOS'
PIO_VAL = BASE / 'AgroUTMACH_PIX' / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data' / 'images' / 'val'
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap07'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_EXP = DIR_VAL / 'cap07_sobremuestreo'
for d in (DIR_FIG, DIR_VAL, DIR_EXP):
    d.mkdir(parents=True, exist_ok=True)
for p in (DATOS, DIR_MOD, PIO_VAL):
    print('OK   ' if p.exists() else 'FALTA', p)

LADO, LOTE, EPOCAS, PACIENCIA = 384, 8, 60, 15          # configuración del NB04 del proyecto para YOLO11m
SEMILLAS = [2026, 2027, 2028]

import torch, ultralytics
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support
HAY_GPU = torch.cuda.is_available()
DISP = 0 if HAY_GPU else 'cpu'
GPU = torch.cuda.get_device_name(0) if HAY_GPU else 'sin GPU'
print('GPU:', GPU, '| PyTorch', torch.__version__, '| Ultralytics', ultralytics.__version__)
assert HAY_GPU or not EN_COLAB, 'Active una GPU en Entorno de ejecución > Cambiar tipo de entorno.'

# Copia local de las 452 imágenes procesadas: evita leer de Drive en cada predicción.
RAIZ = LOCAL / 'PALMA_cap07'
if RAIZ.exists():
    shutil.rmtree(RAIZ)
shutil.copytree(DATOS, RAIZ)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)


ERRORES = {}
CLASES_ORD = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']       # orden alfabético, el de Ultralytics
RUTAS = {p: sorted(q for q in (RAIZ / p).glob('*/*') if q.suffix.lower() in EXT_IMG) for p in ('train', 'val', 'test')}
print({p: len(v) for p, v in RUTAS.items()})
assert len(RUTAS['val']) == 70 and len(RUTAS['test']) == 70
MODELOS, PRED = {}, None


def seccion(nombre, fn):
    # Ejecuta una sección y, si falla, anota el error y deja que las demás continúen.
    t0 = time.time()
    try:
        fn()
        print(f'[{nombre}] terminada en {time.time() - t0:.0f} s')
    except Exception:
        ERRORES[nombre] = traceback.format_exc()[-1500:]
        print(f'[{nombre}] ERROR\n{ERRORES[nombre]}')
    plt.close('all')


def predecir(modelo, rutas, disp=None):
    # Probabilidades por imagen, con las columnas en el orden de CLASES_ORD.
    res = modelo.predict(source=[str(p) for p in rutas], imgsz=LADO, batch=LOTE, device=DISP if disp is None else disp, verbose=False)
    nombres = {int(k): v for k, v in res[0].names.items()}
    P = np.array([r.probs.data.detach().cpu().numpy().astype(float) for r in res])
    return P[:, [next(i for i, n in nombres.items() if n == c) for c in CLASES_ORD]]


def tabla_pred(P, rutas, **extra):
    k = P.argmax(1)
    t = pd.DataFrame({'imagen': [p.name for p in rutas], 'real': [p.parent.name for p in rutas],
                      'pred': [CLASES_ORD[i] for i in k], 'confianza': P.max(1)})
    for i, c in enumerate(CLASES_ORD):
        t['p_' + c] = P[:, i]
    return t.assign(**extra)


def metricas(t):
    pr, se, f1, n = precision_recall_fscore_support(t['real'], t['pred'], labels=CLASES_ORD, zero_division=0)
    return {'exactitud': float(accuracy_score(t['real'], t['pred'])), 'f1_macro': float(f1.mean()),
            'f1_ponderado': float(f1_score(t['real'], t['pred'], labels=CLASES_ORD, average='weighted', zero_division=0)),
            'precision': dict(zip(CLASES_ORD, map(float, pr))), 'sensibilidad': dict(zip(CLASES_ORD, map(float, se))),
            'f1': dict(zip(CLASES_ORD, map(float, f1))), 'n': dict(zip(CLASES_ORD, map(int, n))),
            'matriz': confusion_matrix(t['real'], t['pred'], labels=CLASES_ORD).tolist()}


def galeria(items, archivo, alto=2.9):
    # Fila de imágenes con sus cinco probabilidades debajo. Cada elemento: img, pie, probs (dict por clase), pred.
    n = len(items)
    fig = plt.figure(figsize=(ANCHO_IN, alto))
    gs = fig.add_gridspec(2, n, wspace=0.14, hspace=0.42, height_ratios=[1.25, 1])
    for k, it in enumerate(items):
        ax = fig.add_subplot(gs[0, k]); ax.imshow(it['img']); sin_ejes(ax)
        ax.set_xlabel(it['pie'], fontsize=6, color=TINTA_2, labelpad=3)
        ax = fig.add_subplot(gs[1, k])
        y = np.arange(len(CLASES))[::-1]
        v = [it['probs'][c] for c in CLASES]
        ax.barh(y, v, 0.62, color=[ACENTO if c == it['pred'] else GRIS_FONDO for c in CLASES])
        for yy, vv in zip(y, v):
            if vv > 0.6:
                ax.text(vv - 0.04, yy, coma(vv, 2), va='center', ha='right', fontsize=5.6, color='white')
            else:
                ax.text(vv + 0.04, yy, coma(vv, 2), va='center', fontsize=5.6)
        ax.set_yticks(y); ax.set_yticklabels([NOMBRE_ES[c] for c in CLASES] if k == 0 else [], fontsize=6)
        ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1]); ax.set_xticklabels(['0', '0,5', '1'], fontsize=6); ax.tick_params(axis='y', length=0)
    guardar(fig, DIR_FIG / archivo); plt.show()


def fila_a_item(f, pie):
    ruta = next(p for p in RUTAS['test'] + RUTAS['val'] if p.name == f['imagen'])
    return {'img': recorte_cuadrado(leer_rgb(ruta, 640), 300), 'pie': pie, 'pred': f['pred'],
            'probs': {c: float(f['p_' + c]) for c in CLASES_ORD}}

## 1. Los nueve modelos del proyecto, imagen por imagen

In [ ]:
def s_modelos():
    global PRED
    for d in sorted(DIR_MOD.iterdir()):
        m = re.fullmatch(r'yolo11([nsm])_cls_seed(\d+)', d.name, flags=re.I)
        if m and (d / 'best.pt').exists():
            MODELOS[(m.group(1).lower(), int(m.group(2)))] = d / 'best.pt'
    print('Modelos encontrados:', sorted(MODELOS))
    assert MODELOS, 'No se encontró ningún modelo yolo11*_cls_seed* en 05_MODELOS.'
    trozos, res = [], {}
    for (tam, sem), ruta in sorted(MODELOS.items()):
        mod = YOLO(str(ruta)); clave = f'{tam}_{sem}'; res[clave] = {'archivo_mb': round(ruta.stat().st_size / 1e6, 2)}
        for part in ('val', 'test'):
            t = tabla_pred(predecir(mod, RUTAS[part]), RUTAS[part], modelo=tam, semilla=sem, particion=part)
            trozos.append(t); res[clave][part] = metricas(t)
        print(clave, '| val F1', round(res[clave]['val']['f1_macro'], 4), '| prueba F1', round(res[clave]['test']['f1_macro'], 4))
        del mod
    PRED = pd.concat(trozos, ignore_index=True)
    PRED.to_csv(DIR_VAL / 'cap07_predicciones_9_modelos.csv', index=False)
    VALORES['modelos'] = res

seccion('modelos', s_modelos)

## 2. Los errores del primer modelo y las imágenes difíciles

In [ ]:
def s_errores():
    clave = ('n', 2026) if ('n', 2026) in MODELOS else sorted(MODELOS)[0]
    t = PRED[(PRED['modelo'] == clave[0]) & (PRED['semilla'] == clave[1]) & (PRED['particion'] == 'test')]
    mal = t[t['real'] != t['pred']].sort_values('confianza', ascending=False).head(5)
    items = [fila_a_item(f, f"Real: {NOMBRE_ES[f['real']].lower()}\nPredicho: {NOMBRE_ES[f['pred']].lower()}") for _, f in mal.iterrows()]
    galeria(items, 'Fig_7_3_errores_primer_modelo.png')
    VALORES['errores_primer_modelo'] = {'modelo': f'{clave[0]}_{clave[1]}', 'n_errores': int((t['real'] != t['pred']).sum()),
                                        'detalle': mal[['imagen', 'real', 'pred', 'confianza'] + ['p_' + c for c in CLASES_ORD]].round(4).to_dict('records')}


def s_dificiles():
    t = PRED[PRED['particion'] == 'test'].copy(); t['fallo'] = t['real'] != t['pred']
    n_mod = t.groupby(['modelo', 'semilla']).ngroups
    g = t.groupby('imagen').agg(real=('real', 'first'), fallos=('fallo', 'sum'), confianza_media=('confianza', 'mean'),
                                mas_votada=('pred', lambda s: s.value_counts().index[0])).reset_index()
    g = g.sort_values(['fallos', 'imagen'], ascending=[False, True])
    g.to_csv(DIR_VAL / 'cap07_dificultad_prueba.csv', index=False)
    top = g[g['fallos'] > 0].head(8)
    fig, ejes = plt.subplots(2, 4, figsize=(ANCHO_IN, 3.9), gridspec_kw={'wspace': 0.06, 'hspace': 0.38})
    for ax in ejes.flat:
        sin_ejes(ax)
    for ax, (_, f) in zip(ejes.flat, top.iterrows()):
        ruta = next(p for p in RUTAS['test'] if p.name == f['imagen'])
        ax.imshow(recorte_cuadrado(leer_rgb(ruta, 640), 300))
        ax.set_xlabel(f"{NOMBRE_ES[f['real']]} · fallan {int(f['fallos'])} de {n_mod}\nMás votada: {NOMBRE_ES[f['mas_votada']].lower()}",
                      fontsize=6, color=TINTA_2, labelpad=3)
    guardar(fig, DIR_FIG / 'Fig_7_5_imagenes_dificiles.png'); plt.show()
    VALORES['dificultad'] = {'n_modelos': int(n_mod), 'distribucion_fallos': {int(k): int(v) for k, v in g['fallos'].value_counts().sort_index().items()},
                             'por_clase_sin_fallos': {c: int(((g['real'] == c) & (g['fallos'] == 0)).sum()) for c in CLASES_ORD},
                             'top': top.round(4).to_dict('records'),
                             'escasas': g[g['real'].isin(['damaged', 'empty'])].round(4).to_dict('records')}
    # Voto por mayoría de los modelos de cada tamaño y de los nueve.
    votos = {}
    for nombre, sub in [('nueve', t)] + [(k, t[t['modelo'] == k]) for k in sorted(t['modelo'].unique())]:
        prom = sub.groupby('imagen')[['p_' + c for c in CLASES_ORD]].mean()
        real = sub.groupby('imagen')['real'].first()
        v = pd.DataFrame({'real': real, 'pred': [CLASES_ORD[i] for i in prom.to_numpy().argmax(1)]})
        votos[nombre] = {k: m for k, m in metricas(v).items() if k in ('exactitud', 'f1_macro', 'f1', 'matriz')}
    VALORES['promedio_de_modelos'] = votos
    print(json.dumps({k: round(v['f1_macro'], 4) for k, v in votos.items()}))

seccion('errores', s_errores)
seccion('dificiles', s_dificiles)

## 3. Lo que el modelo recibe

In [ ]:
def s_entrada():
    tam = []
    for part, rutas in RUTAS.items():
        for p in rutas:
            with Image.open(p) as im:
                tam.append((part, p.name, im.size[0], im.size[1]))
    tam = pd.DataFrame(tam, columns=['particion', 'imagen', 'ancho', 'alto'])
    tam['perdida'] = 1 - tam[['ancho', 'alto']].min(axis=1) / tam[['ancho', 'alto']].max(axis=1)
    VALORES['entrada'] = {'n': int(len(tam)), 'cuadradas': int((tam['perdida'] < 0.01).sum()),
                          'perdida_media': float(tam['perdida'].mean()), 'perdida_mediana': float(tam['perdida'].median()),
                          'perdida_max': float(tam['perdida'].max()), 'mas_de_20': int((tam['perdida'] > 0.20).sum()),
                          'verticales': int((tam['alto'] > tam['ancho']).sum()), 'horizontales': int((tam['ancho'] > tam['alto']).sum()),
                          'lado_mayor_mediano': float(tam[['ancho', 'alto']].max(axis=1).median())}
    prueba = tam[tam['particion'] == 'test'].sort_values(['perdida', 'imagen'], ascending=[False, True])
    ruta = next(p for p in RUTAS['test'] if p.name == prueba.iloc[0]['imagen'])
    img = leer_rgb(ruta); h, w = img.shape[:2]; m = min(h, w); y0, x0 = (h - m) // 2, (w - m) // 2
    # La transformación real de Ultralytics, si puede importarse; si no, su equivalente manual.
    origen = 'manual'
    try:
        from ultralytics.data.augment import classify_transforms
        tensor = classify_transforms(size=LADO)(Image.fromarray(img))
        visto = np.clip(tensor.permute(1, 2, 0).numpy(), 0, 1); origen = 'ultralytics.classify_transforms'
        assert visto.shape[:2] == (LADO, LADO) and visto.max() > 0.2
    except Exception as e:
        print('Transformación de Ultralytics no disponible:', repr(e)[:120]); origen = 'manual'
        visto = cv2.resize(img[y0:y0 + m, x0:x0 + m], (LADO, LADO), interpolation=cv2.INTER_AREA) / 255.0
    fig, ejes = plt.subplots(1, 2, figsize=(ANCHO_IN, 2.7), gridspec_kw={'width_ratios': [w / h, 1], 'wspace': 0.08})
    ejes[0].imshow(img); ejes[0].add_patch(Rectangle((x0, y0), m, m, fill=False, ec='white', lw=1.6, ls='--'))
    ejes[0].set_title('a) Fotografía', loc='left', fontweight='bold', fontsize=7)
    ejes[0].set_xlabel(f'{w} × {h} píxeles', fontsize=6.5, color=TINTA_2)
    ejes[1].imshow(visto); ejes[1].set_title('b) Entrada del modelo', loc='left', fontweight='bold', fontsize=7)
    ejes[1].set_xlabel(f'{LADO} × {LADO} píxeles', fontsize=6.5, color=TINTA_2)
    for ax in ejes:
        sin_ejes(ax)
    guardar(fig, DIR_FIG / 'Fig_7_6_entrada_del_modelo.png'); plt.show()
    # Comprobación: predecir sobre el recorte central debe dar casi lo mismo que sobre la fotografía entera.
    clave = ('m', 2026) if ('m', 2026) in MODELOS else sorted(MODELOS)[0]
    mod = YOLO(str(MODELOS[clave])); tmp = LOCAL / 'cap07_tmp' / 'recorte'; tmp.mkdir(parents=True, exist_ok=True)
    rutas = RUTAS['test']; rec = []
    for p in rutas:
        a = leer_rgb(p); hh, ww = a.shape[:2]; mm = min(hh, ww); yy, xx = (hh - mm) // 2, (ww - mm) // 2
        q = tmp / (p.stem + '.png'); Image.fromarray(a[yy:yy + mm, xx:xx + mm]).save(q); rec.append(q)
    Pa, Pb = predecir(mod, rutas), predecir(mod, rec)
    VALORES['entrada'].update({'imagen_figura': ruta.name, 'ancho': int(w), 'alto': int(h), 'transformacion': origen,
                               'recorte_dif_max': float(np.abs(Pa - Pb).max()), 'recorte_dif_media': float(np.abs(Pa - Pb).mean()),
                               'recorte_cambia_clase': int((Pa.argmax(1) != Pb.argmax(1)).sum())})
    print(json.dumps(VALORES['entrada'], ensure_ascii=False))

seccion('entrada', s_entrada)

## 4. Clasificar una imagen nueva: respuesta y tiempo

In [ ]:
def cronometrar(ruta_modelo, disp, rutas, calentamiento=5):
    # Tiempo total por imagen, del archivo a la respuesta, de una en una.
    mod = YOLO(str(ruta_modelo))
    for p in rutas[:calentamiento]:
        mod.predict(source=str(p), imgsz=LADO, device=disp, verbose=False)
    t = []
    for p in rutas:
        t0 = time.perf_counter(); mod.predict(source=str(p), imgsz=LADO, device=disp, verbose=False); t.append((time.perf_counter() - t0) * 1000)
    return {'mediana_ms': float(np.median(t)), 'media_ms': float(np.mean(t)), 'p95_ms': float(np.percentile(t, 95)), 'n': len(t)}


def s_demo():
    clave = ('m', 2026) if ('m', 2026) in MODELOS else sorted(MODELOS)[0]
    t = PRED[(PRED['modelo'] == clave[0]) & (PRED['semilla'] == clave[1]) & (PRED['particion'] == 'test')]
    items, det = [], []
    for c in CLASES:
        sub = t[(t['real'] == c) & (t['pred'] == c)].sort_values('confianza')
        if sub.empty:
            sub = t[t['real'] == c].sort_values('confianza')
        f = sub.iloc[len(sub) // 2]
        items.append(fila_a_item(f, f"Real: {NOMBRE_ES[f['real']].lower()}\nPredicho: {NOMBRE_ES[f['pred']].lower()}"))
        det.append({'imagen': f['imagen'], 'real': f['real'], 'pred': f['pred'], 'confianza': round(float(f['confianza']), 4)})
    galeria(items, 'Fig_7_7_inferencia.png')
    tiempos = {}
    for tam in ('n', 's', 'm'):
        if (tam, 2026) in MODELOS:
            tiempos[tam] = {'cpu': cronometrar(MODELOS[(tam, 2026)], 'cpu', RUTAS['test'][:30])}
            if HAY_GPU:
                tiempos[tam]['gpu'] = cronometrar(MODELOS[(tam, 2026)], 0, RUTAS['test'])
            print(tam, {k: round(v['mediana_ms'], 1) for k, v in tiempos[tam].items()})
    cpu = platform.processor()
    try:
        cpu = next(l.split(':')[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name'))
    except Exception:
        pass
    VALORES['demo'] = {'modelo': f'{clave[0]}_{clave[1]}', 'ejemplos': det, 'tiempos': tiempos, 'cpu': cpu,
                       'nucleos': os.cpu_count(), 'hilos_torch': int(torch.get_num_threads()) if hasattr(torch, 'get_num_threads') else None}

seccion('demo', s_demo)

## 5. Imágenes que no son racimos

In [ ]:
def s_fuera():
    clave = ('m', 2026) if ('m', 2026) in MODELOS else sorted(MODELOS)[0]
    mod = YOLO(str(MODELOS[clave])); tmp = LOCAL / 'cap07_tmp' / 'fuera'; tmp.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(2026); casos = []
    pollos = sorted(p for p in PIO_VAL.iterdir() if p.suffix.lower() in EXT_IMG) if PIO_VAL.exists() else []
    if pollos:
        casos.append(('Galpón de pollos', leer_rgb(pollos[0], 1280)))
    base = leer_rgb(sorted((RAIZ / 'test' / 'ripe').glob('*.jpg'))[0]); h, w = base.shape[:2]; r = int(min(h, w) * 0.22)
    casos.append(('Esquina de una foto', base[:r, :r]))
    casos.append(('Gris uniforme', np.full((LADO, LADO, 3), 128, np.uint8)))
    casos.append(('Ruido', rng.integers(0, 256, (LADO, LADO, 3), dtype=np.uint8)))
    rutas = []
    for k, (nombre, img) in enumerate(casos):
        q = tmp / f'fuera_{k}.png'; Image.fromarray(img).save(q); rutas.append(q)
    P = predecir(mod, rutas); items, det = [], []
    for (nombre, img), p in zip(casos, P):
        pred = CLASES_ORD[int(p.argmax())]
        items.append({'img': recorte_cuadrado(img, 300), 'pie': f'{nombre}\nPredicho: {NOMBRE_ES[pred].lower()}', 'pred': pred,
                      'probs': dict(zip(CLASES_ORD, map(float, p)))})
        det.append({'caso': nombre, 'pred': pred, 'probabilidades': {c: round(float(v), 4) for c, v in zip(CLASES_ORD, p)}})
    galeria(items, 'Fig_7_8_fuera_de_dominio.png')
    # Lo mismo con 60 fotogramas de pollos: ¿qué responde el clasificador de racimos?
    resumen = None
    if len(pollos) >= 10:
        Pp = predecir(mod, pollos[:: max(1, len(pollos) // 60)][:60])
        resumen = {'n': int(len(Pp)), 'clases': {c: int((Pp.argmax(1) == i).sum()) for i, c in enumerate(CLASES_ORD)},
                   'confianza_media': float(Pp.max(1).mean()), 'confianza_min': float(Pp.max(1).min()),
                   'con_confianza_mayor_0_9': int((Pp.max(1) > 0.9).sum())}
    VALORES['fuera_de_dominio'] = {'modelo': f'{clave[0]}_{clave[1]}', 'casos': det, 'pollos': resumen}
    print(json.dumps(VALORES['fuera_de_dominio'], ensure_ascii=False)[:900])

seccion('fuera', s_fuera)

## 6. Curvas de entrenamiento guardadas por el proyecto

In [ ]:
def s_curvas():
    candidatos = []
    for d in sorted(PALMA.iterdir()):
        if d.is_dir() and not d.name.startswith(('01_', '02_')):
            candidatos += list(d.rglob('results.csv'))
    print('Archivos results.csv encontrados:', len(candidatos))
    trozos, res = [], {}
    for (tam, sem) in sorted(MODELOS):
        patron = re.compile(rf'yolo11{tam}_cls_seed{sem}$', re.I)
        hallados = [p for p in candidatos if any(patron.search(x) for x in p.parts)]
        if not hallados:
            continue
        c = pd.read_csv(hallados[0]); c.columns = [x.strip() for x in c.columns]
        ex = c['metrics/accuracy_top1']
        res[f'{tam}_{sem}'] = {'epocas': int(len(c)), 'mejor_epoca': int(c['epoch'][ex.idxmax()]), 'exactitud_val_max': float(ex.max()),
                               'perdida_final': float(c['train/loss'].iloc[-1]), 'segundos': float(c['time'].iloc[-1]),
                               'ruta': str(hallados[0].relative_to(PALMA))}
        trozos.append(c.assign(modelo=tam, semilla=sem))
    if trozos:
        pd.concat(trozos, ignore_index=True).to_csv(DIR_VAL / 'cap07_curvas.csv', index=False)
    VALORES['curvas'] = res
    print(json.dumps(res, ensure_ascii=False)[:900])

seccion('curvas', s_curvas)

## 7. Experimento: sobremuestrear las clases escasas

El entrenamiento del proyecto ve 137 racimos maduros por cada 7 vacíos. Aquí se repiten las imágenes de entrenamiento de las clases con menos de 20 ejemplos hasta acercarlas a 45, se entrena YOLO11m con la configuración del proyecto y se compara, con las mismas semillas y las mismas 70 imágenes de prueba, con los modelos del proyecto. Validación y prueba no se tocan.

In [ ]:
def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def s_sobremuestreo():
    destino = LOCAL / 'PALMA_SOBRE'
    if destino.exists():
        shutil.rmtree(destino)
    shutil.copytree(RAIZ, destino)
    cuenta = {}
    for d in sorted((destino / 'train').iterdir()):
        orig = sorted(p for p in d.iterdir() if p.suffix.lower() in EXT_IMG); n = len(orig)
        veces = max(1, round(45 / n)) if n < 20 else 1
        for p in orig:
            for k in range(1, veces):
                shutil.copy(p, d / f'{p.stem}_rep{k}{p.suffix}')
        cuenta[d.name] = {'original': n, 'veces': veces, 'final': n * veces}
    print(cuenta)
    corridas = []
    for sem in SEMILLAS:
        nombre = f'sobre_seed{sem}'; rj, rc = DIR_EXP / f'{nombre}.json', DIR_EXP / f'{nombre}_pred.csv'
        if rj.exists() and rc.exists():
            print('REUTILIZADA:', nombre); corridas.append(json.loads(rj.read_text(encoding='utf-8'))); continue
        fijar_semilla(sem); t0 = time.time()
        YOLO('yolo11m-cls.pt').train(data=str(destino), epochs=EPOCAS, imgsz=LADO, batch=LOTE, patience=PACIENCIA, device=DISP, workers=2,
                                     seed=sem, deterministic=True, pretrained=True, optimizer='auto', amp=True, cache=True,
                                     project=str(LOCAL / 'cap07_runs'), name=nombre, exist_ok=True, plots=False, verbose=False)
        minutos = (time.time() - t0) / 60
        mejor = LOCAL / 'cap07_runs' / nombre / 'weights' / 'best.pt'
        assert mejor.exists(), f'No se generó {mejor}'
        mod = YOLO(str(mejor)); trozos = []; r = {'semilla': sem, 'minutos': round(minutos, 2)}
        for part in ('val', 'test'):
            t = tabla_pred(predecir(mod, RUTAS[part]), RUTAS[part], semilla=sem, particion=part); trozos.append(t); r[part] = metricas(t)
        rcsv = LOCAL / 'cap07_runs' / nombre / 'results.csv'
        r['epocas'] = int(len(pd.read_csv(rcsv))) if rcsv.exists() else None
        pd.concat(trozos).to_csv(rc, index=False); rj.write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding='utf-8')
        print(f"FINALIZADA: {nombre} | {minutos:.1f} min | val F1 {r['val']['f1_macro']:.4f} | prueba F1 {r['test']['f1_macro']:.4f}")
        corridas.append(r); del mod; gc.collect()
        if HAY_GPU:
            torch.cuda.empty_cache()
    base = {s: VALORES.get('modelos', {}).get(f'm_{s}') for s in SEMILLAS}
    filas = []
    for r in corridas:
        b = base.get(r['semilla'])
        filas.append({'semilla': r['semilla'], 'epocas': r['epocas'], 'minutos': r['minutos'],
                      'sobre_val_f1': r['val']['f1_macro'], 'sobre_prueba_f1': r['test']['f1_macro'], 'sobre_exactitud': r['test']['exactitud'],
                      'base_val_f1': b['val']['f1_macro'] if b else None, 'base_prueba_f1': b['test']['f1_macro'] if b else None,
                      **{f'sobre_f1_{c}': r['test']['f1'][c] for c in CLASES_ORD},
                      **{f'base_f1_{c}': (b['test']['f1'][c] if b else None) for c in CLASES_ORD}})
    tabla = pd.DataFrame(filas); tabla.to_csv(DIR_EXP / 'resumen_sobremuestreo.csv', index=False)
    print(tabla.round(3).to_string(index=False))
    VALORES['sobremuestreo'] = {'entrenamiento': cuenta, 'corridas': corridas, 'resumen': filas,
                                'media_sobre_prueba_f1': float(tabla['sobre_prueba_f1'].mean()),
                                'media_base_prueba_f1': float(tabla['base_prueba_f1'].mean()) if tabla['base_prueba_f1'].notna().all() else None}

seccion('sobremuestreo', s_sobremuestreo)

## 8. Guardar y descargar

In [ ]:
VALORES['errores'] = ERRORES
VALORES['entorno'] = {'gpu': GPU, 'torch': torch.__version__, 'ultralytics': ultralytics.__version__}
ruta_json = DIR_VAL / 'valores_cap07.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
ruta_zip = LOCAL / 'AgroVision_cap07.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Fig_7_*.png')):
        z.write(f, 'cap07/' + f.name)
    for f in [ruta_json] + sorted(DIR_VAL.glob('cap07_*.csv')) + sorted(DIR_EXP.glob('*')):
        if f.is_file():
            z.write(f, f.name)
print('Secciones con error:', list(ERRORES) or 'ninguna')
print('Figuras:', len(FIGURAS), '· zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))